# E-89 HW03 Problem 3: Fashion MNIST Classifier

**Jazmyn Stokes**

This notebook trains a fully-connected neural network to classify Fashion MNIST images. It combines the nine numbered scripts from the repo (`01_setup.py` through `09_evaluate.py`) into one notebook that reads start to finish, with each script's step as its own numbered section below.

## 1. Setup

Imports, device selection, random seed, and matplotlib defaults used throughout the rest of the notebook.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchmetrics
import matplotlib.pyplot as plt
from torchvision import datasets
from torchvision.transforms import v2
from torch.utils.data import random_split, DataLoader

# --- Pick the best available device: cuda > mps > cpu ---
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# --- Seed everything for reproducibility ---
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# --- Sensible matplotlib defaults ---
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["font.size"] = 11

print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device}")

PyTorch version: 2.14.0+cu130
Using device: cpu


## 2. Load Data

Load Fashion MNIST via torchvision, convert to float tensors scaled to [0, 1] with `transforms.v2`, and split the 60k training images into 55k train / 5k validation.

In [2]:
DATA_DIR = "datasets"

transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
])

train_data_full = datasets.FashionMNIST(
    root=DATA_DIR, train=True, download=True, transform=transform
)
test_data = datasets.FashionMNIST(
    root=DATA_DIR, train=False, download=True, transform=transform
)

class_names = train_data_full.classes

# Re-seed right before the random split so it's reproducible on its own.
torch.manual_seed(SEED)
train_data, val_data = random_split(train_data_full, [55000, 5000])

print(f"Train size: {len(train_data)}")
print(f"Validation size: {len(val_data)}")
print(f"Test size: {len(test_data)}")
print(f"Class names: {class_names}")

Train size: 55000
Validation size: 5000
Test size: 10000
Class names: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


## 3. DataLoaders

Wrap the three splits in DataLoaders (`batch_size=32`), shuffling only the training set.

In [3]:
BATCH_SIZE = 32

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

sample_images, sample_labels = next(iter(train_loader))
sample_image, sample_label = sample_images[0], sample_labels[0]

print(f"Sample image shape: {sample_image.shape}")
print(f"Sample image dtype: {sample_image.dtype}")
print(f"Sample label: {sample_label.item()} ({class_names[sample_label.item()]})")

Sample image shape: torch.Size([1, 28, 28])
Sample image dtype: torch.float32
Sample label: 3 (Dress)


## 4. Model

An MLP: `Flatten -> Linear(784,300) -> ReLU -> Linear(300,100) -> ReLU -> Linear(100,10)`. No activation on the final layer, since `nn.CrossEntropyLoss` expects raw logits.

In [4]:
class ImageClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(28 * 28, 300)
        self.linear2 = nn.Linear(300, 100)
        self.linear3 = nn.Linear(100, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.linear1(x))
        x = F.relu(self.linear2(x))
        x = self.linear3(x)  # raw logits, no activation
        return x


model = ImageClassifier().to(device)
loss_fn = nn.CrossEntropyLoss()

num_params = sum(p.numel() for p in model.parameters())
print(f"Model: {model}")
print(f"Total trainable parameters: {num_params:,}")

Model: ImageClassifier(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear1): Linear(in_features=784, out_features=300, bias=True)
  (linear2): Linear(in_features=300, out_features=100, bias=True)
  (linear3): Linear(in_features=100, out_features=10, bias=True)
)
Total trainable parameters: 266,610


## 5. Training Functions

`evaluate()` runs one pass over a loader under `no_grad` and returns the average loss and metric. `train2()` trains for `n_epochs` and returns a history dict (`train_losses`, `train_metrics`, `val_losses`, `val_metrics`) rather than just printing, since the history is plotted later.

In [5]:
def evaluate(model, loader, metric, device):
    model.eval()
    metric.reset()
    total_loss = 0.0
    n_batches = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            loss = loss_fn(logits, y)
            total_loss += loss.item()
            n_batches += 1
            metric.update(logits, y)
    avg_loss = total_loss / n_batches
    metric_value = metric.compute().item()
    return avg_loss, metric_value


def train2(model, train_loader, val_loader, optimizer, loss_fn, metric, device, n_epochs):
    history = {
        "train_losses": [],
        "train_metrics": [],
        "val_losses": [],
        "val_metrics": [],
    }

    for epoch in range(n_epochs):
        model.train()
        metric.reset()
        running_loss = 0.0
        n_batches = 0

        for X, y in train_loader:
            X, y = X.to(device), y.to(device)

            optimizer.zero_grad()
            logits = model(X)
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            n_batches += 1
            metric.update(logits, y)

        train_loss = running_loss / n_batches
        train_metric = metric.compute().item()

        val_loss, val_metric = evaluate(model, val_loader, metric, device)

        history["train_losses"].append(train_loss)
        history["train_metrics"].append(train_metric)
        history["val_losses"].append(val_loss)
        history["val_metrics"].append(val_metric)

        print(
            f"Epoch {epoch + 1}/{n_epochs} - "
            f"train_loss: {train_loss:.4f}, train_acc: {train_metric:.4f}, "
            f"val_loss: {val_loss:.4f}, val_acc: {val_metric:.4f}"
        )

    return history

## 6. Run Training

Train for 20 epochs with plain SGD (`lr=0.1`) and a `torchmetrics` multiclass accuracy metric on the same device as the model.

In [6]:
N_EPOCHS = 20
LEARNING_RATE = 0.1

optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)
metric = torchmetrics.classification.MulticlassAccuracy(num_classes=10).to(device)

history = train2(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    metric=metric,
    device=device,
    n_epochs=N_EPOCHS,
)

Epoch 1/20 - train_loss: 0.6026, train_acc: 0.7801, val_loss: 0.4444, val_acc: 0.8421


Epoch 2/20 - train_loss: 0.4041, train_acc: 0.8503, val_loss: 0.3782, val_acc: 0.8634


Epoch 3/20 - train_loss: 0.3611, train_acc: 0.8659, val_loss: 0.3822, val_acc: 0.8597


Epoch 4/20 - train_loss: 0.3327, train_acc: 0.8780, val_loss: 0.3444, val_acc: 0.8748


Epoch 5/20 - train_loss: 0.3128, train_acc: 0.8837, val_loss: 0.3840, val_acc: 0.8598


...

Epoch 16/20 - train_loss: 0.2088, train_acc: 0.9210, val_loss: 0.3704, val_acc: 0.8799


Epoch 17/20 - train_loss: 0.2036, train_acc: 0.9224, val_loss: 0.3108, val_acc: 0.8915


Epoch 18/20 - train_loss: 0.1959, train_acc: 0.9243, val_loss: 0.3466, val_acc: 0.8865


Epoch 19/20 - train_loss: 0.1938, train_acc: 0.9255, val_loss: 0.3283, val_acc: 0.8887


Epoch 20/20 - train_loss: 0.1860, train_acc: 0.9285, val_loss: 0.3338, val_acc: 0.8880


## 7. Plot Accuracy and Loss

Training accuracy for epoch *i* is an average of batch accuracies computed while the weights were still changing across that epoch, so it's plotted at the epoch midpoint (i - 0.5). Validation accuracy, computed with the epoch's final weights, is plotted at the epoch's end (i).

In [7]:
n_epochs = len(history["train_metrics"])
epochs = list(range(1, n_epochs + 1))
train_x = [e - 0.5 for e in epochs]  # epoch midpoints

# --- Figure 1: accuracy ---
plt.figure()
plt.plot(train_x, history["train_metrics"], label="Training accuracy", marker="o")
plt.plot(epochs, history["val_metrics"], label="Validation accuracy", marker="o")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs. Validation Accuracy")
plt.ylim(0, 1)
plt.legend()
plt.show()

print(f"Final training accuracy:   {history['train_metrics'][-1]:.4f}")
print(f"Final validation accuracy: {history['val_metrics'][-1]:.4f}")

# --- Figure 2: training loss ---
plt.figure()
plt.plot(epochs, history["train_losses"], label="Training loss", color="tab:red", marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.legend()
plt.show()

Final training accuracy:   0.9285
Final validation accuracy: 0.8880


## 8. Predictions

Run the trained model on 3 validation images: print predicted vs. actual class names, then softmax the logits and show the top-4 predicted probabilities per image. `round()` isn't implemented on MPS, so tensors are moved to `cpu` before rounding. Finally, show the 3 images with their labels.

In [8]:
N_SAMPLES = 3

model.eval()
images, labels = next(iter(val_loader))
images, labels = images[:N_SAMPLES], labels[:N_SAMPLES]

with torch.no_grad():
    logits = model(images.to(device))

predicted = logits.argmax(dim=1)

# round() isn't implemented on MPS -- move to cpu first.
probs = F.softmax(logits, dim=1).cpu()
predicted_cpu = predicted.cpu()

print("Predictions:")
for i in range(N_SAMPLES):
    actual_name = class_names[labels[i].item()]
    predicted_name = class_names[predicted_cpu[i].item()]
    print(f"  Image {i}: predicted = {predicted_name!r}, actual = {actual_name!r}")

print("\nTop-4 predicted probabilities per image:")
top4_probs, top4_idx = torch.topk(probs, 4, dim=1)
for i in range(N_SAMPLES):
    print(f"  Image {i}:")
    for prob, idx in zip(top4_probs[i], top4_idx[i]):
        print(f"    {class_names[idx.item()]:<12s} {round(prob.item(), 4)}")

# --- Show the images with predicted vs. actual labels ---
fig, axes = plt.subplots(1, N_SAMPLES, figsize=(9, 3))
for i, ax in enumerate(axes):
    ax.imshow(images[i].squeeze(), cmap="gray")
    actual_name = class_names[labels[i].item()]
    predicted_name = class_names[predicted_cpu[i].item()]
    ax.set_title(f"Pred: {predicted_name}\nActual: {actual_name}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

Predictions:
  Image 0: predicted = 'Sneaker', actual = 'Sneaker'
  Image 1: predicted = 'Coat', actual = 'Coat'
  Image 2: predicted = 'Pullover', actual = 'Pullover'

Top-4 predicted probabilities per image:
  Image 0:
    Sneaker      0.987
    Ankle boot   0.0125
    Sandal       0.0004
    Trouser      0.0001
  Image 1:
    Coat         0.9922
    Pullover     0.0074
    Shirt        0.0004
    T-shirt/top  0.0
  Image 2:
    Pullover     0.8502
    Shirt        0.1174
    Coat         0.0318
    T-shirt/top  0.0006


## 9. Final Evaluation

One final run on the held-out test set.

In [9]:
test_loss, test_accuracy = evaluate(model, test_loader, metric, device)

print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Total trainable parameters: {num_params:,}")

Test accuracy: 0.8871
Total trainable parameters: 266,610


## Conclusion

After training the 3-layer MLP (784 -> 300 -> 100 -> 10, 266,610 parameters) for 20 epochs with plain SGD (lr=0.1) on Fashion MNIST:

- **Final validation accuracy: 88.80%**
- **Final test accuracy: 88.71%**

The close agreement between validation and test accuracy suggests the model generalizes consistently and isn't overfit to the validation split. Training accuracy (92.85%) is noticeably higher than both, which is expected given no regularization (dropout, weight decay, or data augmentation) was used -- the model has enough capacity to fit the training set somewhat more tightly than it generalizes.